In [14]:
# remove duplicate rows from logged injuries
import pandas as pd
pd.read_csv('nfl_injuries_cleaned_with_logged_id.csv').drop_duplicates().to_csv('nfl_injuries_cleaned_with_logged_id.csv', index=False)

In [ ]:
import pandas as pd
import pymysql
import csv
import config

conn = pymysql.connect(
    host=config.host,
    port=config.port,
    user=config.user,
    passwd=config.passwd,
    db=config.db,
    autocommit=True
)

cur = conn.cursor(pymysql.cursors.DictCursor)

with open('F_injury_events.csv', 'r') as f:
    df_events = [{k: str(v) for k, v in row.items()} for row in csv.DictReader(f, skipinitialspace=True)]

with open('F_logged_injuries_with_id.csv', 'r') as f:
    df_logs = [{k: str(v) for k, v in row.items()} for row in csv.DictReader(f, skipinitialspace=True)]

with open('F_nfl_injuries_cleaned_with_logged_id.csv', 'r') as f:
    df_injuries = [{k: str(v) for k, v in row.items()} for row in csv.DictReader(f, skipinitialspace=True)]

with open('pbp_cleaned.csv', 'r') as f:
    df_pbp = [{k: str(v) for k, v in row.items()} for row in csv.DictReader(f, skipinitialspace=True)]

with open('nfl_players.csv', 'r') as f: 
    df_players = [{k: str(v) for k, v in row.items()} for row in csv.DictReader(f, skipinitialspace=True)]

cur.execute('''DROP TABLE IF EXISTS events''')
cur.execute('''DROP TABLE IF EXISTS logs''')
cur.execute('''DROP TABLE IF EXISTS injuryReport''')
cur.execute('''DROP TABLE IF EXISTS pbp''')
cur.execute('''DROP TABLE IF EXISTS players''')


def to_int(v):
    if v is None:
        return None
    s = str(v).strip()
    if s in ('', 'NA', 'NA ', 'nan', '<NA>', 'None'):
        return None
    try:
        return int(float(s))
    except Exception:
        return None


def to_float(v):
    if v is None:
        return None
    s = str(v).strip()
    if s in ('', 'NA', 'NA ', 'nan', '<NA>', 'None'):
        return None
    try:
        return float(s)
    except Exception:
        return None


def to_text(v):
    if v is None:
        return None
    s = str(v).strip()
    if s in ('', 'NA', 'NA ', 'nan', '<NA>', 'None'):
        return None
    return s


sql_logs = '''CREATE TABLE IF NOT EXISTS logs (
    log_id BIGINT AUTO_INCREMENT,
    gameId VARCHAR(50),
    playId INT,
    gameDate DATE,
    home_team VARCHAR(10),
    away_team VARCHAR(10),
    description TEXT,
    injury_text TEXT,
    playerName VARCHAR(50),
    number VARCHAR(10),
    team VARCHAR(10),
    return_status VARCHAR(20),
    team_source VARCHAR(25),
    first_initial VARCHAR(5),
    last_name VARCHAR(50),
    next_game_date DATE,
    PRIMARY KEY (log_id),
    INDEX idx_logs_event (gameId, playId, playerName)
);'''

sql_injuries = '''CREATE TABLE IF NOT EXISTS injuryReport (
    injury_report_row_id BIGINT AUTO_INCREMENT,
    injuryId VARCHAR(100),
    transactionDate DATE,
    team VARCHAR(10),
    full_name VARCHAR(50),
    season INT,
    week INT,
    game_type VARCHAR(25),
    action VARCHAR(100) NOT NULL DEFAULT 'Unknown',
    injury_type VARCHAR(255),
    severity VARCHAR(10),
    injury_detail VARCHAR(15),
    surgery_mention INT,
    position VARCHAR(10),
    source VARCHAR(15),
    prev_date DATE,
    days_since_prev INT,
    new_injury_flag BOOLEAN,
    on_ir BOOLEAN,
    prev_on_ir BOOLEAN,
    prev_season INT,
    injury_split_reason VARCHAR(25),
    notes VARCHAR(255),
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    gsis_id VARCHAR(25),
    report_primary_injury VARCHAR(255),
    report_secondary_injury VARCHAR(255),
    report_status VARCHAR(25),
    practice_primary_injury VARCHAR(255),
    practice_secondary_injury VARCHAR(255),
    practice_status VARCHAR(50),
    date_modified VARCHAR(25),
    first_initial VARCHAR(1),
    loggedInGame INT,
    PRIMARY KEY (injury_report_row_id),
    INDEX idx_injuryId (injuryId)
);'''

sql_pbp = '''CREATE TABLE IF NOT EXISTS pbp (
    playId INT,
    gameId VARCHAR(25),
    old_game_id VARCHAR(25),
    home_team VARCHAR(25),
    away_team VARCHAR(25),
    season_type VARCHAR(25),
    week INT,
    yardline_100 INT,
    gameDate DATE,
    qtr INT,
    description TEXT,
    play_type VARCHAR(25),
    down_num INT,
    goal_to_go INT,
    game_clock VARCHAR(25),
    yrdln VARCHAR(10),
    ydstogo INT,
    ydsnet INT,
    epa FLOAT(10,3),
    season INT,
    start_time VARCHAR(25),
    stadium VARCHAR(255),
    weather VARCHAR(255),
    nfl_api_id VARCHAR(255),
    away_score INT,
    home_score INT,
    location VARCHAR(25),
    spread_line FLOAT(6,2),
    total_line FLOAT(6,2),
    div_game INT,
    roof VARCHAR(25),
    surface VARCHAR(25),
    temp INT,
    wind VARCHAR(25),
    home_coach VARCHAR(255),
    away_coach VARCHAR(255),
    stadium_id VARCHAR(255),
    game_stadium VARCHAR(255),
    PRIMARY KEY (gameId, playId)
);'''

sql_events = '''CREATE TABLE IF NOT EXISTS events (
    injuryId VARCHAR(100),
    playerName VARCHAR(50),
    first_report_date DATE,
    injury_type VARCHAR(255),
    injury_detail VARCHAR(50),
    gameId VARCHAR(25),
    playId INT,
    confidence_level VARCHAR(20),
    confidence_score FLOAT(5,2),
    loggedInGame INT,
    PRIMARY KEY (injuryId),
    FOREIGN KEY (injuryId) REFERENCES injuryReport(injuryId),
    FOREIGN KEY (gameId, playId) REFERENCES pbp(gameId, playId),
    FOREIGN KEY (gameId, playId) REFERENCES logs(gameId, playId)
);'''

cur.execute(sql_logs)
cur.execute(sql_injuries)
cur.execute(sql_pbp)
cur.execute(sql_events)


sql = '''INSERT INTO logs (
    gameId, playId, gameDate, home_team, away_team, description, injury_text,
    playerName, number, team, return_status, team_source, first_initial, last_name, next_game_date
) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)'''

block_size = 500
tokens = []
n = 0

for row in df_logs:
    tokens.append([
        to_text(row.get('game_id')),
        to_int(row.get('play_id')),
        to_text(row.get('game_date')),
        to_text(row.get('home_team')),
        to_text(row.get('away_team')),
        to_text(row.get('desc')),
        to_text(row.get('injury_text')),
        to_text(row.get('player_name')),
        to_text(row.get('player_number')),
        to_text(row.get('team')),
        to_text(row.get('return_status')),
        to_text(row.get('team_source')),
        to_text(row.get('first_initial')),
        to_text(row.get('last_name')),
        to_text(row.get('next_game_date')),
    ])
    if len(tokens) >= block_size:
        cur.executemany(sql, tokens)
        tokens = []
    n += 1

if len(tokens) > 0:
    cur.executemany(sql, tokens)

print(n)


sql = '''INSERT INTO injuryReport (
    injuryId, transactionDate, team, full_name, season, week, game_type, action,
    injury_type, severity, injury_detail, surgery_mention, position, source,
    prev_date, days_since_prev, new_injury_flag, on_ir, prev_on_ir, prev_season,
    injury_split_reason, notes, first_name, last_name, gsis_id, report_primary_injury,
    report_secondary_injury, report_status, practice_primary_injury, practice_secondary_injury,
    practice_status, date_modified, first_initial, loggedInGame
) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)'''

tokens = []
n = 0

for row in df_injuries:
    tokens.append([
        to_text(row.get('injury_id')),
        to_text(row.get('date')),
        to_text(row.get('team')),
        to_text(row.get('full_name')),
        to_int(row.get('season')),
        to_int(row.get('week')),
        to_text(row.get('game_type')),
        to_text(row.get('action')) or 'Unknown',
        to_text(row.get('injury_type')),
        to_text(row.get('severity')),
        to_text(row.get('injury_detail')),
        to_int(row.get('surgery_mention')),
        to_text(row.get('position')),
        to_text(row.get('source')),
        to_text(row.get('prev_date')),
        to_int(row.get('days_since_prev')),
        to_int(row.get('new_injury_flag')),
        to_int(row.get('on_ir')),
        to_int(row.get('prev_on_ir')),
        to_int(row.get('prev_season')),
        to_text(row.get('injury_split_reason')),
        to_text(row.get('notes')),
        to_text(row.get('first_name')),
        to_text(row.get('last_name')),
        to_text(row.get('gsis_id')),
        to_text(row.get('report_primary_injury')),
        to_text(row.get('report_secondary_injury')),
        to_text(row.get('report_status')),
        to_text(row.get('practice_primary_injury')),
        to_text(row.get('practice_secondary_injury')),
        to_text(row.get('practice_status')),
        to_text(row.get('date_modified')),
        to_text(row.get('first_initial')),
        to_int(row.get('loggedInGame')),
    ])

    if len(tokens) >= block_size:
        cur.executemany(sql, tokens)
        tokens = []
    n += 1

if len(tokens) > 0:
    cur.executemany(sql, tokens)

print(n)


sql = '''INSERT INTO pbp (
    playId, gameId, old_game_id, home_team, away_team, season_type, week, yardline_100,
    gameDate, qtr, description, play_type, down_num, goal_to_go, game_clock, yrdln,
    ydstogo, ydsnet, epa, season, start_time, stadium, weather, nfl_api_id,
    away_score, home_score, location, spread_line, total_line, div_game, roof,
    surface, temp, wind, home_coach, away_coach, stadium_id, game_stadium
) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)'''

tokens = []
n = 0

for row in df_pbp:
    tokens.append([
        to_int(row.get('play_id')),
        to_text(row.get('game_id')),
        to_text(row.get('old_game_id')),
        to_text(row.get('home_team')),
        to_text(row.get('away_team')),
        to_text(row.get('season_type')),
        to_int(row.get('week')),
        to_int(row.get('yardline_100')),
        to_text(row.get('game_date')),
        to_int(row.get('qtr')),
        to_text(row.get('desc')),
        to_text(row.get('play_type')),
        to_int(row.get('down')),
        to_int(row.get('goal_to_go')),
        to_text(row.get('time')),
        to_text(row.get('yrdln')),
        to_int(row.get('ydstogo')),
        to_int(row.get('ydsnet')),
        to_float(row.get('epa')),
        to_int(row.get('season')),
        to_text(row.get('start_time')),
        to_text(row.get('stadium')),
        to_text(row.get('weather')),
        to_text(row.get('nfl_api_id')),
        to_int(row.get('away_score')),
        to_int(row.get('home_score')),
        to_text(row.get('location')),
        to_float(row.get('spread_line')),
        to_float(row.get('total_line')),
        to_int(row.get('div_game')),
        to_text(row.get('roof')),
        to_text(row.get('surface')),
        to_int(row.get('temp')),
        to_text(row.get('wind')),
        to_text(row.get('home_coach')),
        to_text(row.get('away_coach')),
        to_text(row.get('stadium_id')),
        to_text(row.get('game_stadium')),
    ])
    if len(tokens) >= block_size:
        cur.executemany(sql, tokens)
        tokens = []
    n += 1

if len(tokens) > 0:
    cur.executemany(sql, tokens)

print(n)


sql = '''INSERT INTO events (
    injuryId, playerName, first_report_date, injury_type, injury_detail,
    gameId, playId, confidence_level, confidence_score, loggedInGame
) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)'''

cur.execute('SELECT DISTINCT injuryId FROM injuryReport')
valid_injury_ids = {to_text(r['injuryId']) for r in cur.fetchall() if to_text(r.get('injuryId')) is not None}

tokens = []
n = 0
skipped_events = 0

for row in df_events:
    injury_id = to_text(row.get('injury_id'))
    if injury_id not in valid_injury_ids:
        skipped_events += 1
        continue
    tokens.append([
        injury_id,
        to_text(row.get('player_name')),
        to_text(row.get('first_report_date')),
        to_text(row.get('injury_type')),
        to_text(row.get('injury_detail')),
        to_text(row.get('game_id')),
        to_int(row.get('play_id')),
        to_text(row.get('confidence_level')),
        to_float(row.get('confidence_score')),
        to_int(row.get('loggedInGame')),
    ])
    if len(tokens) >= block_size:
        cur.executemany(sql, tokens)
        tokens = []
    n += 1

if len(tokens) > 0:
    cur.executemany(sql, tokens)

print(n)
print(f'Skipped events with missing parent injuryId: {skipped_events}')

14531
98518
723252
51260
Skipped events with missing parent injuryId: 19


In [4]:
import csv
import pymysql
import config


def to_int_nullable(v):
    if v is None:
        return None
    s = str(v).strip()
    if s in ('', 'NA', 'NA ', 'nan', '<NA>', 'None'):
        return None
    try:
        return int(float(s))
    except Exception:
        return None


def to_text_nullable(v):
    if v is None:
        return None
    s = str(v).strip()
    if s in ('', 'NA', 'NA ', 'nan', '<NA>', 'None'):
        return None
    return s


conn_players = pymysql.connect(
    host=config.host,
    port=config.port,
    user=config.user,
    passwd=config.passwd,
    db=config.db,
    autocommit=True
)
cur_players = conn_players.cursor(pymysql.cursors.DictCursor)

with open('nfl_players.csv', 'r', encoding='utf-8') as f:
    df_players = [{k: str(v) for k, v in row.items()} for row in csv.DictReader(f, skipinitialspace=True)]

cur_players.execute('''DROP TABLE IF EXISTS players''')

sql_players = '''CREATE TABLE IF NOT EXISTS players (
    gsis_id VARCHAR(25),
    display_name VARCHAR(100),
    common_first_name VARCHAR(50),
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    short_name VARCHAR(50),
    football_name VARCHAR(50),
    suffix VARCHAR(20),
    esb_id VARCHAR(25),
    nfl_id VARCHAR(25),
    pfr_id VARCHAR(25),
    pff_id VARCHAR(25),
    otc_id VARCHAR(25),
    espn_id VARCHAR(25),
    smart_id VARCHAR(50),
    birth_date DATE,
    position_group VARCHAR(25),
    position VARCHAR(25),
    ngs_position_group VARCHAR(50),
    ngs_position VARCHAR(50),
    height INT,
    weight INT,
    headshot TEXT,
    college_name VARCHAR(255),
    college_conference VARCHAR(255),
    jersey_number INT,
    rookie_season INT,
    last_season INT,
    latest_team VARCHAR(10),
    status VARCHAR(25),
    ngs_status VARCHAR(25),
    ngs_status_short_description VARCHAR(100),
    years_of_experience INT,
    pff_position VARCHAR(25),
    pff_status VARCHAR(25),
    draft_year INT,
    draft_round INT,
    draft_pick INT,
    draft_team VARCHAR(10),
    PRIMARY KEY (gsis_id),
    INDEX idx_players_display_name (display_name),
    INDEX idx_players_last_first (last_name, first_name),
    INDEX idx_players_latest_team (latest_team)
 );'''

cur_players.execute(sql_players)

sql = '''INSERT INTO players (
    gsis_id, display_name, common_first_name, first_name, last_name, short_name, football_name, suffix,
    esb_id, nfl_id, pfr_id, pff_id, otc_id, espn_id, smart_id, birth_date,
    position_group, position, ngs_position_group, ngs_position, height, weight, headshot,
    college_name, college_conference, jersey_number, rookie_season, last_season, latest_team, status,
    ngs_status, ngs_status_short_description, years_of_experience, pff_position, pff_status,
    draft_year, draft_round, draft_pick, draft_team
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)'''

block_size = 500
tokens = []
inserted_players = 0
missing_gsis_id = 0
duplicate_gsis_ids = 0
seen_gsis_ids = set()

for row in df_players:
    gsis_id = to_text_nullable(row.get('gsis_id'))
    if gsis_id is None:
        missing_gsis_id += 1
        continue
    if gsis_id in seen_gsis_ids:
        duplicate_gsis_ids += 1
        continue

    seen_gsis_ids.add(gsis_id)
    tokens.append([
        gsis_id,
        to_text_nullable(row.get('display_name')),
        to_text_nullable(row.get('common_first_name')),
        to_text_nullable(row.get('first_name')),
        to_text_nullable(row.get('last_name')),
        to_text_nullable(row.get('short_name')),
        to_text_nullable(row.get('football_name')),
        to_text_nullable(row.get('suffix')),
        to_text_nullable(row.get('esb_id')),
        to_text_nullable(row.get('nfl_id')),
        to_text_nullable(row.get('pfr_id')),
        to_text_nullable(row.get('pff_id')),
        to_text_nullable(row.get('otc_id')),
        to_text_nullable(row.get('espn_id')),
        to_text_nullable(row.get('smart_id')),
        to_text_nullable(row.get('birth_date')),
        to_text_nullable(row.get('position_group')),
        to_text_nullable(row.get('position')),
        to_text_nullable(row.get('ngs_position_group')),
        to_text_nullable(row.get('ngs_position')),
        to_int_nullable(row.get('height')),
        to_int_nullable(row.get('weight')),
        to_text_nullable(row.get('headshot')),
        to_text_nullable(row.get('college_name')),
        to_text_nullable(row.get('college_conference')),
        to_int_nullable(row.get('jersey_number')),
        to_int_nullable(row.get('rookie_season')),
        to_int_nullable(row.get('last_season')),
        to_text_nullable(row.get('latest_team')),
        to_text_nullable(row.get('status')),
        to_text_nullable(row.get('ngs_status')),
        to_text_nullable(row.get('ngs_status_short_description')),
        to_int_nullable(row.get('years_of_experience')),
        to_text_nullable(row.get('pff_position')),
        to_text_nullable(row.get('pff_status')),
        to_int_nullable(row.get('draft_year')),
        to_int_nullable(row.get('draft_round')),
        to_int_nullable(row.get('draft_pick')),
        to_text_nullable(row.get('draft_team')),
    ])

    if len(tokens) >= block_size:
        cur_players.executemany(sql, tokens)
        tokens = []
    inserted_players += 1

if len(tokens) > 0:
    cur_players.executemany(sql, tokens)

print(f'Inserted players: {inserted_players}')
print(f'Skipped players with missing gsis_id: {missing_gsis_id}')
print(f'Skipped duplicate gsis_id rows: {duplicate_gsis_ids}')

cur_players.close()
conn_players.close()

Inserted players: 24356
Skipped players with missing gsis_id: 0
Skipped duplicate gsis_id rows: 0


In [7]:
import pymysql
import config

conn_bridge = pymysql.connect(
    host=config.host,
    port=config.port,
    user=config.user,
    passwd=config.passwd,
    db=config.db,
    autocommit=True
)
cur_bridge = conn_bridge.cursor(pymysql.cursors.DictCursor)

required_tables = ['players', 'injuryReport', 'events']
missing_tables = []
for table_name in required_tables:
    cur_bridge.execute("SHOW TABLES LIKE %s", (table_name,))
    if cur_bridge.fetchone() is None:
        missing_tables.append(table_name)

if missing_tables:
    raise RuntimeError(f"Missing required tables: {missing_tables}. Run the load cells first.")

cur_bridge.execute("SELECT COUNT(*) AS n FROM players")
players_count = cur_bridge.fetchone()['n']
if players_count == 0:
    raise RuntimeError("players table is empty. Run Cell 3 first to load nfl_players.csv.")

cur_bridge.execute("""
    SELECT COUNT(*) AS n
    FROM injuryReport
    WHERE injuryId IS NULL
       OR TRIM(injuryId) = ''
       OR LOWER(TRIM(injuryId)) = 'na'
""")
invalid_injury_ids = cur_bridge.fetchone()['n'] or 0
print(f"injuryReport rows with invalid injuryId (excluded from bridge): {invalid_injury_ids}")

cur_bridge.execute('''DROP TABLE IF EXISTS event_player_bridge''')
cur_bridge.execute('''DROP TABLE IF EXISTS injury_player_bridge''')
cur_bridge.execute('''DROP TABLE IF EXISTS injury_master''')

sql_injury_master = '''CREATE TABLE IF NOT EXISTS injury_master (
    injuryId VARCHAR(100) NOT NULL,
    has_injury_report BOOLEAN NOT NULL DEFAULT 0,
    has_event_row BOOLEAN NOT NULL DEFAULT 0,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (injuryId)
 );'''
cur_bridge.execute(sql_injury_master)

sql_injury_master_insert = '''INSERT INTO injury_master (injuryId, has_injury_report, has_event_row)
SELECT
    u.injuryId,
    MAX(CASE WHEN u.src = 'injuryReport' THEN 1 ELSE 0 END) AS has_injury_report,
    MAX(CASE WHEN u.src = 'events' THEN 1 ELSE 0 END) AS has_event_row
FROM (
    SELECT DISTINCT TRIM(ir.injuryId) AS injuryId, 'injuryReport' AS src
    FROM injuryReport ir
    WHERE ir.injuryId IS NOT NULL
      AND TRIM(ir.injuryId) <> ''
      AND LOWER(TRIM(ir.injuryId)) <> 'na'
    UNION ALL
    SELECT DISTINCT TRIM(e.injuryId) AS injuryId, 'events' AS src
    FROM events e
    WHERE e.injuryId IS NOT NULL
      AND TRIM(e.injuryId) <> ''
      AND LOWER(TRIM(e.injuryId)) <> 'na'
 ) u
GROUP BY u.injuryId
'''
cur_bridge.execute(sql_injury_master_insert)
injury_master_rows = cur_bridge.rowcount

sql_injury_bridge = '''CREATE TABLE IF NOT EXISTS injury_player_bridge (
    injuryId VARCHAR(100) NOT NULL,
    resolved_gsis_id VARCHAR(25),
    match_method VARCHAR(40) NOT NULL,
    match_score DECIMAL(5,2) NOT NULL DEFAULT 0.00,
    candidate_count INT NOT NULL DEFAULT 0,
    is_manual_override BOOLEAN NOT NULL DEFAULT 0,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    season_hint INT,
    position_hint VARCHAR(25),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (injuryId),
    INDEX idx_bridge_gsis (resolved_gsis_id),
    INDEX idx_bridge_method (match_method),
    CONSTRAINT fk_injury_bridge_master
      FOREIGN KEY (injuryId) REFERENCES injury_master(injuryId)
      ON DELETE CASCADE ON UPDATE CASCADE,
    CONSTRAINT fk_injury_bridge_player
      FOREIGN KEY (resolved_gsis_id) REFERENCES players(gsis_id)
      ON DELETE SET NULL ON UPDATE CASCADE
 );'''
cur_bridge.execute(sql_injury_bridge)

sql_gold = '''INSERT INTO injury_player_bridge (
    injuryId, resolved_gsis_id, match_method, match_score, candidate_count,
    first_name, last_name, season_hint, position_hint
)
SELECT
    x.injuryId,
    x.only_gsis_id,
    'gsis_exact',
    1.00,
    1,
    x.first_name,
    x.last_name,
    x.season_hint,
    x.position_hint
FROM (
    SELECT
        ir.injuryId,
        MIN(CASE
            WHEN ir.gsis_id IS NOT NULL
             AND TRIM(ir.gsis_id) <> ''
             AND LOWER(TRIM(ir.gsis_id)) <> 'na'
            THEN TRIM(ir.gsis_id)
        END) AS only_gsis_id,
        COUNT(DISTINCT CASE
            WHEN ir.gsis_id IS NOT NULL
             AND TRIM(ir.gsis_id) <> ''
             AND LOWER(TRIM(ir.gsis_id)) <> 'na'
            THEN TRIM(ir.gsis_id)
        END) AS gsis_count,
        MIN(NULLIF(TRIM(ir.first_name), '')) AS first_name,
        MIN(NULLIF(TRIM(ir.last_name), '')) AS last_name,
        MIN(ir.season) AS season_hint,
        MIN(NULLIF(TRIM(ir.position), '')) AS position_hint
    FROM injuryReport ir
    WHERE ir.injuryId IS NOT NULL
      AND TRIM(ir.injuryId) <> ''
      AND LOWER(TRIM(ir.injuryId)) <> 'na'
    GROUP BY ir.injuryId
) x
JOIN players p
  ON p.gsis_id = x.only_gsis_id
WHERE x.gsis_count = 1
'''
cur_bridge.execute(sql_gold)
gold_inserted = cur_bridge.rowcount

sql_silver = '''INSERT INTO injury_player_bridge (
    injuryId, resolved_gsis_id, match_method, match_score, candidate_count,
    first_name, last_name, season_hint, position_hint
)
SELECT
    cand.injuryId,
    cand.resolved_gsis_id,
    'name_season_unique',
    0.85,
    cand.candidate_count,
    cand.first_name,
    cand.last_name,
    cand.season_hint,
    cand.position_hint
FROM (
    SELECT
        base.injuryId,
        MIN(p.gsis_id) AS resolved_gsis_id,
        COUNT(DISTINCT p.gsis_id) AS candidate_count,
        base.first_name,
        base.last_name,
        base.season_hint,
        base.position_hint
    FROM (
        SELECT
            ir.injuryId,
            MIN(NULLIF(TRIM(ir.first_name), '')) AS first_name,
            MIN(NULLIF(TRIM(ir.last_name), '')) AS last_name,
            MIN(ir.season) AS season_hint,
            MIN(NULLIF(TRIM(ir.position), '')) AS position_hint
        FROM injuryReport ir
        WHERE ir.injuryId IS NOT NULL
          AND TRIM(ir.injuryId) <> ''
          AND LOWER(TRIM(ir.injuryId)) <> 'na'
        GROUP BY ir.injuryId
    ) base
    LEFT JOIN injury_player_bridge b
      ON b.injuryId = base.injuryId
    JOIN players p
      ON LOWER(TRIM(p.first_name)) = LOWER(base.first_name)
     AND LOWER(TRIM(p.last_name)) = LOWER(base.last_name)
     AND (
         base.season_hint IS NULL
         OR (
             (p.rookie_season IS NULL OR base.season_hint >= p.rookie_season)
             AND (p.last_season IS NULL OR base.season_hint <= p.last_season)
         )
     )
     AND (
         base.position_hint IS NULL
         OR TRIM(base.position_hint) = ''
         OR p.position IS NULL
         OR TRIM(p.position) = ''
         OR UPPER(TRIM(p.position)) = UPPER(TRIM(base.position_hint))
     )
    WHERE b.injuryId IS NULL
      AND base.first_name IS NOT NULL
      AND base.last_name IS NOT NULL
    GROUP BY base.injuryId, base.first_name, base.last_name, base.season_hint, base.position_hint
    HAVING COUNT(DISTINCT p.gsis_id) = 1
) cand
'''
cur_bridge.execute(sql_silver)
silver_inserted = cur_bridge.rowcount

sql_bronze = '''INSERT INTO injury_player_bridge (
    injuryId, resolved_gsis_id, match_method, match_score, candidate_count,
    first_name, last_name, season_hint, position_hint
)
SELECT
    cand.injuryId,
    cand.resolved_gsis_id,
    'initial_last_season_unique',
    0.65,
    cand.candidate_count,
    cand.first_name,
    cand.last_name,
    cand.season_hint,
    cand.position_hint
FROM (
    SELECT
        base.injuryId,
        MIN(p.gsis_id) AS resolved_gsis_id,
        COUNT(DISTINCT p.gsis_id) AS candidate_count,
        base.first_name,
        base.last_name,
        base.season_hint,
        base.position_hint
    FROM (
        SELECT
            ir.injuryId,
            MIN(NULLIF(TRIM(ir.first_initial), '')) AS first_initial,
            MIN(NULLIF(TRIM(ir.first_name), '')) AS first_name,
            MIN(NULLIF(TRIM(ir.last_name), '')) AS last_name,
            MIN(ir.season) AS season_hint,
            MIN(NULLIF(TRIM(ir.position), '')) AS position_hint
        FROM injuryReport ir
        WHERE ir.injuryId IS NOT NULL
          AND TRIM(ir.injuryId) <> ''
          AND LOWER(TRIM(ir.injuryId)) <> 'na'
        GROUP BY ir.injuryId
    ) base
    LEFT JOIN injury_player_bridge b
      ON b.injuryId = base.injuryId
    JOIN players p
      ON UPPER(LEFT(TRIM(p.first_name), 1)) = UPPER(base.first_initial)
     AND LOWER(TRIM(p.last_name)) = LOWER(base.last_name)
     AND (
         base.season_hint IS NULL
         OR (
             (p.rookie_season IS NULL OR base.season_hint >= p.rookie_season)
             AND (p.last_season IS NULL OR base.season_hint <= p.last_season)
         )
     )
     AND (
         base.position_hint IS NULL
         OR TRIM(base.position_hint) = ''
         OR p.position IS NULL
         OR TRIM(p.position) = ''
         OR UPPER(TRIM(p.position)) = UPPER(TRIM(base.position_hint))
     )
    WHERE b.injuryId IS NULL
      AND base.first_initial IS NOT NULL
      AND base.last_name IS NOT NULL
    GROUP BY base.injuryId, base.first_name, base.last_name, base.season_hint, base.position_hint
    HAVING COUNT(DISTINCT p.gsis_id) = 1
) cand
'''
cur_bridge.execute(sql_bronze)
bronze_inserted = cur_bridge.rowcount

sql_unresolved = '''INSERT INTO injury_player_bridge (
    injuryId, resolved_gsis_id, match_method, match_score, candidate_count,
    first_name, last_name, season_hint, position_hint
)
SELECT
    im.injuryId,
    NULL,
    'unresolved',
    0.00,
    0,
    NULL,
    NULL,
    NULL,
    NULL
FROM injury_master im
LEFT JOIN injury_player_bridge b
  ON b.injuryId = im.injuryId
WHERE b.injuryId IS NULL
'''
cur_bridge.execute(sql_unresolved)
unresolved_inserted = cur_bridge.rowcount

sql_event_bridge = '''CREATE TABLE IF NOT EXISTS event_player_bridge (
    injuryId VARCHAR(100) NOT NULL,
    gameId VARCHAR(25),
    playId INT,
    resolved_gsis_id VARCHAR(25),
    match_method VARCHAR(40) NOT NULL,
    match_score DECIMAL(5,2) NOT NULL DEFAULT 0.00,
    loggedInGame INT,
    PRIMARY KEY (injuryId),
    INDEX idx_event_bridge_gsis (resolved_gsis_id),
    INDEX idx_event_bridge_gameplay (gameId, playId),
    INDEX idx_event_bridge_method (match_method),
    CONSTRAINT fk_event_bridge_events
      FOREIGN KEY (injuryId) REFERENCES events(injuryId)
      ON DELETE CASCADE ON UPDATE CASCADE,
    CONSTRAINT fk_event_bridge_injury_bridge
      FOREIGN KEY (injuryId) REFERENCES injury_player_bridge(injuryId)
      ON DELETE CASCADE ON UPDATE CASCADE,
    CONSTRAINT fk_event_bridge_player
      FOREIGN KEY (resolved_gsis_id) REFERENCES players(gsis_id)
      ON DELETE SET NULL ON UPDATE CASCADE
 );'''
cur_bridge.execute(sql_event_bridge)

sql_event_insert = '''INSERT INTO event_player_bridge (
    injuryId, gameId, playId, resolved_gsis_id, match_method, match_score, loggedInGame
)
SELECT
    e.injuryId,
    e.gameId,
    e.playId,
    b.resolved_gsis_id,
    COALESCE(b.match_method, 'unresolved') AS match_method,
    COALESCE(b.match_score, 0.00) AS match_score,
    e.loggedInGame
FROM events e
LEFT JOIN injury_player_bridge b
  ON b.injuryId = e.injuryId
WHERE e.injuryId IS NOT NULL
  AND TRIM(e.injuryId) <> ''
  AND LOWER(TRIM(e.injuryId)) <> 'na'
'''
cur_bridge.execute(sql_event_insert)
event_bridge_inserted = cur_bridge.rowcount

cur_bridge.execute("SELECT COUNT(*) AS n FROM injury_master")
injury_master_total = cur_bridge.fetchone()['n']

cur_bridge.execute("SELECT COUNT(*) AS total, SUM(CASE WHEN resolved_gsis_id IS NOT NULL THEN 1 ELSE 0 END) AS linked FROM injury_player_bridge")
injury_bridge_cov = cur_bridge.fetchone()

cur_bridge.execute("SELECT COUNT(*) AS total, SUM(CASE WHEN resolved_gsis_id IS NOT NULL THEN 1 ELSE 0 END) AS linked FROM event_player_bridge")
event_bridge_cov = cur_bridge.fetchone()

cur_bridge.execute("SELECT match_method, COUNT(*) AS n FROM injury_player_bridge GROUP BY match_method ORDER BY n DESC")
method_counts = cur_bridge.fetchall()

print('Bridge build complete.')
print(f'injury_master rows: {injury_master_total}')
print(f'Inserted by method -> gold: {gold_inserted}, silver: {silver_inserted}, bronze: {bronze_inserted}, unresolved: {unresolved_inserted}')
print(f'event_player_bridge rows: {event_bridge_inserted}')
print('')
print('Injury bridge coverage:')
print(f"linked {injury_bridge_cov['linked']} / {injury_bridge_cov['total']}")
print('Event bridge coverage:')
print(f"linked {event_bridge_cov['linked']} / {event_bridge_cov['total']}")
print('')
print('Method distribution:')
for row in method_counts:
    print(f"{row['match_method']}: {row['n']}")

cur_bridge.close()
conn_bridge.close()

injuryReport rows with invalid injuryId (excluded from bridge): 7
Bridge build complete.
injury_master rows: 51260
Inserted by method -> gold: 35805, silver: 7, bronze: 11218, unresolved: 4230
event_player_bridge rows: 51260

Injury bridge coverage:
linked 47030 / 51260
Event bridge coverage:
linked 47030 / 51260

Method distribution:
gsis_exact: 35805
initial_last_season_unique: 11218
unresolved: 4230
name_season_unique: 7
